In [1]:
import requests

In [2]:
url = requests.get("http://www.gutenberg.org/files/1112/1112.txt")
text = url.text

In [3]:
text

'<!DOCTYPE html>\n<html class="client-nojs" lang="en-US" dir="ltr">\n<head>\n <meta charset="UTF-8" >\n\n<title>404 | Project Gutenberg</title>\n <link rel="stylesheet" href="/gutenberg/gutenberg-globals.css?v=4">\n\n <meta name="viewport" content="width=device-width, initial-scale=1">\n <meta name="keywords" content="books, ebooks, free, kindle, android, iphone, ipad">\n <meta name="google-site-verification" content="wucOEvSnj5kP3Ts_36OfP64laakK-1mVTg-ptrGC9io">\n <meta name="alexaVerifyID" content="4WNaCljsE-A82vP_ih2H_UqXZvM">\n \n <link rel="copyright" href="https://www.gnu.org/copyleft/fdl.html">\n <link rel="icon" type="image/png" href="/gutenberg/favicon.ico" sizes="16x16" >\n \n <meta property="og:title"        content="Project Gutenberg" >\n <meta property="og:type"         content="website" >\n <meta property="og:url"          content="https://www.gutenberg.org/" >\n <meta property="og:description"  content="Project Gutenberg is a library of free eBooks." >\n <meta property="

In [4]:
def read_and_return_common_words(url: str = 'https://www.gutenberg.org/cache/epub/1513/pg1513.txt', n: int = 10) -> list[tuple[str, int]]:
    text: str = requests.get(url).text

    # Gutenberg books are always contained in between these START-END statements
    start: int = text.find('*** START OF THE PROJECT GUTENBERG EBOOK ROMEO AND JULIET ***')
    end: int = text.find('*** END OF THE PROJECT GUTENBERG EBOOK ROMEO AND JULIET ***')
    text = text[start:end].lower()

    translator: dict = str.maketrans('', '', string.punctuation + string.digits) # maketrans(x, y, z) -> x: chars to replace (None); y: replace with (None); y: chars to delete (all chars from string.punctuation and string.digits)
    words: list[str] = text.translate(translator).split()

    word_count: dict[str, int] = {}
    for word in words:
        word_count[word] = word_count.get(word, 0) + 1

    return sorted(word_count.items(), key=lambda x: x[1], reverse=True)[:n]

In [5]:
def analyze_cat_data(url: str = 'https://api.thecatapi.com/v1/breeds') -> dict:
    cats: list[dict] = requests.get(url).json()

    # i.
    weights: list = []
    for cat in cats:
        metric_str: str = cat['weight']['metric'] # weight is also a dict with keys 'metric' and 'imperial'
        low, high = [float(weight_val.strip()) for weight_val in metric_str.split('-')] # weight values are separated by "-" so we save both values as low and high by using split("-") and then we strip() bc there are spaces (["3 ", " 5"])
        avg_weight: float = (low + high) / 2
        weights.append(avg_weight)

    weight_stats: dict[str, float] = {
        'min': min(weights),
        'max': max(weights),
        'mean': arithmetic.calculate_mean(weights),
        'median': arithmetic.calculate_median(weights),
        'std': arithmetic.calculate_std(weights)
    }

    # ii.
    lifespans: list[float] = []
    for cat in cats:
        life_str: str = cat['life_span']
        low, high = [float(life_val.strip()) for life_val in life_str.split('-')]
        avg_life: float = (low + high) / 2
        lifespans.append(avg_life)

    lifespan_stats: dict[str, float] = {
        'min': min(lifespans),
        'max': max(lifespans),
        'mean': arithmetic.calculate_mean(lifespans),
        'median': arithmetic.calculate_median(lifespans),
        'std': arithmetic.calculate_std(lifespans)
    }

    # iii.
    freq_table: dict[str, list[str]] = {}
    for cat in cats:
        country: str = cat.get('origin', 'Unknown')
        breed: str = cat['name']
        if country not in freq_table:
            freq_table[country] = []
        freq_table[country].append(breed)

    return {
        'weight': weight_stats,
        'lifespan': lifespan_stats,
        'frequency_table': freq_table
    }
